In [20]:
import polars as pl
import plotly.express as px

In [21]:
df = pl.read_parquet("../data/processed/clean_tripdata.parquet")
df.glimpse()

Rows: 6006904
Columns: 26
$ ride_id                       <str> 'B7516B7B2708CC24', 'BCF70C0B42130B92', 'E004845CDF26C9D4', 'CE6ACBC2A319AA59', '12F3FE6984F93383', '6B3160CE448781BA', 'FB10CC36BED3C86B', '4D47190D0D5E6E55', '02531D76F32E6F6B', '20E8890ED8FCBF1B'
$ rideable_type                 <str> 'classic_bike', 'electric_bike', 'classic_bike', 'classic_bike', 'electric_bike', 'electric_bike', 'electric_bike', 'classic_bike', 'classic_bike', 'electric_bike'
$ started_at           <datetime[μs]> 2025-10-05 15:26:37.782000, 2026-09-11 02:06:11.044000, 2025-10-14 17:05:58.193000, 2026-05-27 10:27:11.879000, 2026-08-24 05:37:01.378000, 2025-10-04 11:53:40.309000, 2026-05-18 12:49:36.303000, 2026-05-24 12:27:14.893000, 2026-06-02 14:15:13.902000, 2026-07-26 15:41:55.611000
$ ended_at             <datetime[μs]> 2025-10-05 15:29:47.109000, 2026-09-11 02:10:50.196000, 2025-10-14 17:14:50.362000, 2026-05-27 10:29:47.582000, 2026-08-24 05:40:45.344000, 2025-10-04 12:15:08.315000, 2026-05-18 1

In [22]:
import plotly.graph_objects as go

SURFACE, INK, INK_2 = "#fcfcfb", "#0b0b0b", "#52514e"
GROUP_LABEL = {"member": "Members", "casual": "Casual riders"}
GROUP_FILL = {"member": "#2a78d6", "casual": "#e34948"}

def tint(hex_color, t):
    """Blend a hex color toward white by t (0 = original, 1 = white)."""
    r, g, b = (int(hex_color[i:i + 2], 16) for i in (1, 3, 5))
    return "#{:02x}{:02x}{:02x}".format(*(round(c + (255 - c) * t) for c in (r, g, b)))

counts = df.group_by("member_casual", "rideable_type").agg(pl.len().alias("trips"))
total = counts["trips"].sum()
group_totals = (
    counts.group_by("member_casual").agg(pl.col("trips").sum())
    .sort("trips", descending=True)
)
pct = {g: n / total * 100 for g, n in group_totals.iter_rows()}

# Hierarchy: rider type (header, full color) -> bike type (lighter tints of the same hue)
ids, labels, parents, values, colors, text_colors, shares = [], [], [], [], [], [], []
for g, g_total in group_totals.iter_rows():
    ids.append(g); labels.append(GROUP_LABEL[g]); parents.append("")
    values.append(g_total); colors.append(GROUP_FILL[g]); text_colors.append("#ffffff")
    shares.append(g_total / total * 100)

    kids = counts.filter(pl.col("member_casual") == g).sort("trips", descending=True)
    for k, (_, bike, n) in enumerate(kids.iter_rows()):
        ids.append(f"{g}/{bike}")
        labels.append(bike.replace("_", " ").capitalize())
        parents.append(g)
        values.append(n)
        colors.append(tint(GROUP_FILL[g], [0.35, 0.55, 0.7, 0.8][min(k, 3)]))
        text_colors.append(INK)
        shares.append(n / total * 100)

fig = go.Figure(
    go.Treemap(
        ids=ids, labels=labels, parents=parents, values=values,
        branchvalues="total",
        customdata=shares,
        texttemplate="<b>%{label}</b><br>%{value:,.0f} trips<br>%{customdata:.1f}%",
        textfont=dict(size=16, color=text_colors),
        marker=dict(colors=colors, line=dict(width=2, color=SURFACE)),
        hovertemplate="<b>%{label}</b><br>%{value:,.0f} trips (%{customdata:.1f}% of all trips)<extra></extra>",
        pathbar=dict(visible=False),
    )
)
fig.update_layout(
    title=dict(
        text=(f"<b>Casual riders account for {pct['casual']:.0f}% of trips; members for {pct['member']:.0f}%</b>"
              f"<br><span style='font-size:13px;color:{INK_2}'>Total trips by rider type and bike type</span>"),
        x=0, xanchor="left", font=dict(size=18, color=INK),
    ),
    paper_bgcolor=SURFACE,
    font=dict(family="system-ui, -apple-system, 'Segoe UI', sans-serif", color=INK_2),
    margin=dict(t=90, l=20, r=20, b=20), height=420,
)
fig.show()
fig.write_image("../figs/01_treemap_rider_and_bike_type.svg", width=1200)

In [23]:
from plotly.subplots import make_subplots

SURFACE, INK, INK_2, MUTED = "#fcfcfb", "#0b0b0b", "#52514e", "#898781"
SEASONS = ["Winter", "Spring", "Summer", "Autumn"]
RIDER_LABEL = {"member": "Members", "casual": "Casual riders"}
RIDER_COLOR = {"member": "#2a78d6", "casual": "#e34948"}

trips = {(s, r): n for s, r, n in df.group_by("season", "member_casual").agg(pl.len()).iter_rows()}
total = {s: trips[s, "member"] + trips[s, "casual"] for s in SEASONS}
casual_share = {s: trips[s, "casual"] / total[s] * 100 for s in SEASONS}

# Headline numbers
x_summer_vs_winter = total["Summer"] / total["Winter"]
drop = {r: (1 - trips["Winter", r] / trips["Summer", r]) * 100 for r in RIDER_LABEL}

fig = make_subplots(
    rows=1, cols=2, column_widths=[0.62, 0.38], horizontal_spacing=0.08,
    subplot_titles=["Rides per season (millions)", "Casual riders' share of each season's rides"],
)
fig.update_annotations(font=dict(size=13, color=INK_2))  # style subplot titles only

# Left: volume per season, stacked by rider type, labeled directly
for r in ["member", "casual"]:
    fig.add_trace(
        go.Bar(
            x=SEASONS, y=[trips[s, r] / 1e6 for s in SEASONS], name=RIDER_LABEL[r],
            marker=dict(color=RIDER_COLOR[r], line=dict(width=2, color=SURFACE)),
            text=[f"{trips[s, r] / 1e6:.2f}M" for s in SEASONS],
            textposition="inside", insidetextanchor="middle",
            textfont=dict(size=12, color="#ffffff"),
            hovertemplate=f"<b>{RIDER_LABEL[r]}</b><br>%{{x}}: %{{y:.2f}}M rides<extra></extra>",
        ),
        row=1, col=1,
    )
for s in SEASONS:
    fig.add_annotation(
        x=s, y=total[s] / 1e6, xref="x", yref="y", text=f"<b>{total[s] / 1e6:.2f}M</b>",
        showarrow=False, yshift=14, font=dict(size=14, color=INK),
    )

# Right: how the mix shifts with the season
fig.add_trace(
    go.Bar(
        x=SEASONS, y=[casual_share[s] for s in SEASONS], showlegend=False,
        marker=dict(color=RIDER_COLOR["casual"], line=dict(width=2, color=SURFACE)),
        text=[f"{casual_share[s]:.0f}%" for s in SEASONS], textposition="outside",
        textfont=dict(size=13, color=INK), cliponaxis=False,
        hovertemplate="%{x}: %{y:.1f}% of rides are casual<extra></extra>",
    ),
    row=1, col=2,
)

fig.update_layout(
    barmode="stack",
    title=dict(
        text=(f"<b>Summer brings {x_summer_vs_winter:.1f}x the rides of winter, "
              "and casual riders are the ones who disappear</b>"
              f"<br><span style='font-size:13px;color:{INK_2}'>"
              f"From summer to winter, casual rides fall {drop['casual']:.0f}% "
              f"versus {drop['member']:.0f}% for members</span>"),
        x=0, xanchor="left", font=dict(size=18, color=INK),
    ),
    paper_bgcolor=SURFACE, plot_bgcolor=SURFACE,
    font=dict(family="system-ui, -apple-system, 'Segoe UI', sans-serif", color=INK_2),
    legend=dict(orientation="h", x=0, y=-0.1, font=dict(size=12, color=INK_2)),
    margin=dict(t=130, l=20, r=20, b=60), height=480,
)
fig.update_yaxes(visible=False, showgrid=False, zeroline=False)
fig.update_yaxes(range=[0, max(total.values()) / 1e6 * 1.15], row=1, col=1)
fig.update_yaxes(range=[0, 50], row=1, col=2)
fig.update_xaxes(showgrid=False, showline=False, ticks="", tickfont=dict(size=13, color=INK_2))
fig.show()
fig.write_image("../figs/02_rides_per_season.svg", width=1200)

In [24]:
from datetime import datetime

SURFACE, INK, INK_2, MUTED = "#fcfcfb", "#0b0b0b", "#52514e", "#898781"
GRID = "#e1e0d9"
RIDER_LABEL = {"member": "Members", "casual": "Casual riders"}
RIDER_COLOR = {"member": "#2a78d6", "casual": "#e34948"}

# Rides per calendar month. The data starts on 30 Sep 2025 (58 rides), so that stub month is dropped.
monthly = (
    df.with_columns(pl.col("started_at").dt.truncate("1mo").alias("ym"))
    .filter(pl.col("ym") >= datetime(2025, 10, 1))
    .group_by("ym", "member_casual").agg(pl.len().alias("trips"))
    .sort("ym")
)
series = {
    g: monthly.filter(pl.col("member_casual") == g) for g in RIDER_LABEL
}
swing = {g: s["trips"].max() / s["trips"].min() for g, s in series.items()}

fig = go.Figure()
for g in ["member", "casual"]:
    s = series[g]
    y = (s["trips"] / 1e6).to_list()
    fig.add_trace(
        go.Scatter(
            x=s["ym"].to_list(), y=y, name=RIDER_LABEL[g], mode="lines+text",
            line=dict(color=RIDER_COLOR[g], width=2.5),
            text=[""] * (len(y) - 1) + [f"<b>{RIDER_LABEL[g]}</b>"],  # label the line at its end
            textposition="middle right", textfont=dict(size=13, color=INK), cliponaxis=False,
            hovertemplate="%{y:.2f}M rides<extra>" + RIDER_LABEL[g] + "</extra>",
        )
    )
fig.update_layout(
    hovermode="x unified",
    title=dict(
        text=(f"<b>Casual ridership swings {swing['casual']:.0f}x between its quietest and busiest month; "
              f"members only {swing['member']:.0f}x</b>"
              f"<br><span style='font-size:13px;color:{INK_2}'>"
              "Rides per month by rider type, Oct 2025 – Sep 2026</span>"),
        x=0, xanchor="left", font=dict(size=18, color=INK),
    ),
    paper_bgcolor=SURFACE, plot_bgcolor=SURFACE,
    font=dict(family="system-ui, -apple-system, 'Segoe UI', sans-serif", color=INK_2),
    legend=dict(orientation="h", x=0, y=-0.15, font=dict(size=12, color=INK_2)),
    margin=dict(t=160, l=50, r=110, b=70), height=510,
)
fig.update_yaxes(rangemode="tozero", ticksuffix="M", showline=False, ticks="",
                 gridcolor=GRID, gridwidth=1, zeroline=False, tickfont=dict(size=12, color=MUTED))
fig.update_xaxes(tickformat="%b '%y", dtick="M1", showgrid=False, showline=True, linecolor=GRID,
                 ticks="", tickfont=dict(size=12, color=INK_2))
# Highlight: vertical markers at casual riders' busiest and quietest month
cs = series["casual"]
hi = cs.filter(pl.col("trips") == pl.col("trips").max())
lo = cs.filter(pl.col("trips") == pl.col("trips").min())
for row, kind in ((hi, "Busiest"), (lo, "Quietest")):
    d, n = row["ym"][0], row["trips"][0]
    fig.add_shape(type="line", x0=d, x1=d, y0=0, y1=1, xref="x", yref="paper",
                 line=dict(color=MUTED, width=1.5, dash="dot"), layer="below")
    fig.add_annotation(x=d, y=1, xref="x", xshift=52, yref="paper", yanchor="bottom", showarrow=False,
                       text=f"{kind} casual month<br><b>{d:%b '%y}: {n / 1e6:.2f}M</b>", align="left",
                       font=dict(size=11, color=INK_2))
fig.update_yaxes(range=[0, max(s["trips"].max() for s in series.values()) / 1e6 * 1.18])
fig.show()
fig.write_image("../figs/03_rides_per_month.svg", width=1200)

In [25]:
SURFACE, INK, INK_2, MUTED = "#fcfcfb", "#0b0b0b", "#52514e", "#898781"
GRID = "#e1e0d9"
RIDER_LABEL = {"member": "Members", "casual": "Casual riders"}
RIDER_COLOR = {"member": "#2a78d6", "casual": "#e34948"}

q = (
    df.group_by("member_casual")
    .agg([(pl.col("ride_length").quantile(p) / 60).alias(f"p{int(p * 100)}") for p in (0.25, 0.5, 0.75, 0.9)])
)
pct = {row["member_casual"]: row for row in q.iter_rows(named=True)}

fig = go.Figure()
for g in ["member", "casual"]:
    p, name, color = pct[g], RIDER_LABEL[g], RIDER_COLOR[g]
    # thin whisker: 25th -> 90th percentile
    fig.add_trace(go.Scatter(
        x=[p["p25"], p["p90"]], y=[name, name], mode="lines", showlegend=False, hoverinfo="skip",
        line=dict(color=color, width=2),
    ))
    # thick bar: 25th -> 75th percentile
    fig.add_trace(go.Scatter(
        x=[p["p25"], p["p75"]], y=[name, name], mode="lines", showlegend=False,
        line=dict(color=color, width=22),
        hovertemplate=(f"<b>{name}</b><br>25% of rides ≤ {p['p25']:.1f} min<br>"
                       f"50% ≤ {p['p50']:.1f} min<br>75% ≤ {p['p75']:.1f} min<br>"
                       f"90% ≤ {p['p90']:.1f} min<extra></extra>"),
    ))
    # median dot, label placed well above the bar
    fig.add_trace(go.Scatter(
        x=[p["p50"]], y=[name], mode="markers", showlegend=False, hoverinfo="skip",
        marker=dict(color="#ffffff", size=11, line=dict(color=color, width=3)),
    ))
    fig.add_annotation(
        x=p["p50"], y=name, xref="x", yref="y", yshift=28, showarrow=False,
        text=f"<b>median {p['p50']:.1f} min</b>", font=dict(size=13, color=INK),
    )
    # quartile labels: below the bar (annotations, so the offset is explicit)
    for key in ("p25", "p75"):
        fig.add_annotation(
            x=p[key], y=name, xref="x", yref="y", yshift=-26, showarrow=False,
            text=f"{p[key]:.1f}", font=dict(size=11, color=MUTED),
        )
    # 90th percentile: right of the end of the line
    fig.add_annotation(
        x=p["p90"], y=name, xref="x", yref="y", xshift=8, xanchor="left", showarrow=False,
        text=f"{p['p90']:.0f} min", font=dict(size=11, color=MUTED),
    )

fig.update_layout(
    title=dict(
        text=(f"<b>Half of casual rides last over {pct['casual']['p50']:.1f} min; "
              f"for members, {pct['member']['p50']:.1f} min</b>"
              f"<br><span style='font-size:13px;color:{INK_2}'>"
              "Ride length. Bar = middle 50% of rides, dot = median, line = up to the 90th percentile</span>"),
        x=0, xanchor="left", font=dict(size=18, color=INK),
    ),
    paper_bgcolor=SURFACE, plot_bgcolor=SURFACE,
    font=dict(family="system-ui, -apple-system, 'Segoe UI', sans-serif", color=INK_2),
    margin=dict(t=110, l=20, r=60, b=60), height=400,
)
fig.update_xaxes(range=[0, max(p["p90"] for p in pct.values()) * 1.2], ticksuffix=" min",
                 gridcolor=GRID, zeroline=False, showline=False, ticks="",
                 tickfont=dict(size=12, color=MUTED))
fig.update_yaxes(categoryorder="array", categoryarray=["Members", "Casual riders"],  # casual on top
                 range=[-0.6, 1.6],  # padding so labels above/below the bars fit
                 showgrid=False, ticks="", showline=False, tickfont=dict(size=14, color=INK))
fig.show()
fig.write_image("../figs/04_ride_length_range.svg", width=1200)

In [26]:
SURFACE, INK, INK_2, MUTED = "#fcfcfb", "#0b0b0b", "#52514e", "#898781"
RIDER_LABEL = {"member": "Members", "casual": "Casual riders"}
RIDER_COLOR = {"member": "#2a78d6", "casual": "#e34948"}

# Round trip = null when it can't be determined, so compare only rides where it is known
known = df.filter(pl.col("is_round_trip").is_not_null())
stats = {
    g: dict(
        known=known.filter(pl.col("member_casual") == g).height,
        round=known.filter((pl.col("member_casual") == g) & pl.col("is_round_trip")).height,
        unknown=df.filter((pl.col("member_casual") == g) & pl.col("is_round_trip").is_null()).height,
        all=df.filter(pl.col("member_casual") == g).height,
    )
    for g in RIDER_LABEL
}
rate = {g: s["round"] / s["known"] * 100 for g, s in stats.items()}
ratio = rate["casual"] / rate["member"]

order = ["member", "casual"]  # horizontal bars draw bottom-up, so casual ends on top
fig = go.Figure(
    go.Bar(
        y=[RIDER_LABEL[g] for g in order], x=[rate[g] for g in order], orientation="h",
        marker=dict(color=[RIDER_COLOR[g] for g in order], line=dict(width=2, color=SURFACE)),
        width=0.5,
        text=[f"<b>{rate[g]:.1f}%</b>  ({stats[g]['round']:,} of {stats[g]['known']:,} rides)" for g in order],
        textposition="outside", cliponaxis=False,
        textfont=dict(size=13, color=INK),
        hovertemplate="%{y}: %{x:.1f}% round trips<extra></extra>",
    )
)
fig.update_layout(
    title=dict(
        text=(f"<b>Casual riders are {ratio:.1f}x as likely to end a ride where they started</b>"
              f"<br><span style='font-size:13px;color:{INK_2}'>"
              "Round trips as % of rides, by rider type</span>"),
        x=0, xanchor="left", font=dict(size=18, color=INK),
    ),
    paper_bgcolor=SURFACE, plot_bgcolor=SURFACE,
    font=dict(family="system-ui, -apple-system, 'Segoe UI', sans-serif", color=INK_2),
    margin=dict(t=100, l=20, r=20, b=70), height=300, showlegend=False,
    annotations=[dict(
        x=0, y=-0.28, xref="paper", yref="paper", showarrow=False, xanchor="left",
        font=dict(size=11, color=MUTED),
        text=("Excludes rides where a round trip can't be determined: "
              + ", ".join(f"{RIDER_LABEL[g].lower()} {stats[g]['unknown'] / stats[g]['all'] * 100:.0f}%" for g in RIDER_LABEL)
              + " of rides."),
    )],
)
fig.update_xaxes(visible=False, range=[0, max(rate.values()) * 3.2])
fig.update_yaxes(showgrid=False, ticks="", showline=False, tickfont=dict(size=14, color=INK))
fig.show()
fig.write_image("../figs/05_round_trips.svg", width=1200)

In [27]:
SURFACE, INK, INK_2, MUTED = "#fcfcfb", "#0b0b0b", "#52514e", "#898781"
RIDER_LABEL = {"member": "Members", "casual": "Casual riders"}
RIDER_COLOR = {"member": "#2a78d6", "casual": "#e34948"}
TOP = 10

def short(name, n=30):
    return name if len(name) <= n else name[: n - 1] + "…"

top, coverage = {}, {}
for g in RIDER_LABEL:
    rides = df.filter(pl.col("member_casual") == g)
    with_station = rides.filter(pl.col("start_station_name").is_not_null())
    coverage[g] = with_station.height / rides.height * 100
    top[g] = (
        with_station.group_by("start_station_name").agg(pl.len().alias("n"))
        .sort("n", descending=True).head(TOP)
    )

fig = make_subplots(
    rows=1, cols=2, horizontal_spacing=0.34,
    subplot_titles=[f"{RIDER_LABEL[g]}: top {TOP} start stations" for g in ["member", "casual"]],
)
fig.update_annotations(font=dict(size=13, color=INK_2))  # style subplot titles only

for i, g in enumerate(["member", "casual"], start=1):
    names = top[g]["start_station_name"].to_list()
    counts = top[g]["n"].to_list()
    fig.add_trace(
        go.Bar(
            x=counts, y=list(range(TOP)), orientation="h", showlegend=False,
            marker=dict(color=RIDER_COLOR[g], line=dict(width=2, color=SURFACE)),
            text=[f"{n / 1000:.1f}k" for n in counts], textposition="outside", cliponaxis=False,
            textfont=dict(size=12, color=INK),
            customdata=names, hovertemplate="<b>%{customdata}</b><br>%{x:,} rides<extra></extra>",
        ),
        row=1, col=i,
    )
    fig.update_yaxes(
        tickmode="array", tickvals=list(range(TOP)), ticktext=[short(n) for n in names],
        autorange="reversed", showgrid=False, ticks="", showline=False,
        tickfont=dict(size=12, color=INK_2), automargin=True, row=1, col=i,
    )
    fig.update_xaxes(visible=False, range=[0, max(counts) * 1.25], row=1, col=i)

fig.update_layout(
    title=dict(
        text=("<b>Casual riders start at the lakefront and tourist spots; members start downtown</b>"
              f"<br><span style='font-size:13px;color:{INK_2}'>"
              "Rides per start station. Only rides with a recorded start station: "
              + ", ".join(f"{coverage[g]:.0f}% of {RIDER_LABEL[g].lower()}" for g in ["member", "casual"])
              + "</span>"),
        x=0, xanchor="left", font=dict(size=18, color=INK),
    ),
    paper_bgcolor=SURFACE, plot_bgcolor=SURFACE,
    font=dict(family="system-ui, -apple-system, 'Segoe UI', sans-serif", color=INK_2),
    margin=dict(t=110, l=20, r=40, b=30), height=500,
)
fig.show()
fig.write_image("../figs/06_top_start_stations.svg", width=1200)

In [28]:
SURFACE, INK, INK_2, MUTED = "#fcfcfb", "#0b0b0b", "#52514e", "#898781"
GRID = "#e1e0d9"
RIDER_LABEL = {"member": "Members", "casual": "Casual riders"}
RIDER_COLOR = {"member": "#2a78d6", "casual": "#e34948"}
DAYS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]  # day_of_week is ISO (1 = Monday)

med = {
    (g, d): m
    for g, d, m in df.group_by("member_casual", "day_of_week")
    .agg((pl.col("ride_length").median() / 60).alias("m"))
    .iter_rows()
}
# weekend uplift: average of Sat/Sun medians minus average of Mon-Fri medians
uplift = {
    g: sum(med[g, d] for d in (6, 7)) / 2 - sum(med[g, d] for d in range(1, 6)) / 5
    for g in RIDER_LABEL
}

fig = go.Figure()
for g in ["member", "casual"]:
    y = [med[g, d] for d in range(1, 8)]
    fig.add_trace(
        go.Scatter(
            x=DAYS, y=y, name=RIDER_LABEL[g], mode="lines+text",
            line=dict(color=RIDER_COLOR[g], width=2.5),
            text=[""] * 6 + [f"<b>{RIDER_LABEL[g]}</b>"],  # label the line at its end
            textposition="middle right", textfont=dict(size=13, color=INK), cliponaxis=False,
            hovertemplate="%{y:.1f} min (median)<extra>" + RIDER_LABEL[g] + "</extra>",
        )
    )
fig.update_layout(
    hovermode="x unified",
    title=dict(
        text=(f"<b>Weekends make a casual ride {uplift['casual']:.1f} min longer; "
              f"a member's only {uplift['member']:.1f} min</b>"
              f"<br><span style='font-size:13px;color:{INK_2}'>"
              "Median ride length by day of the week</span>"),
        x=0, xanchor="left", font=dict(size=18, color=INK),
    ),
    paper_bgcolor=SURFACE, plot_bgcolor=SURFACE,
    font=dict(family="system-ui, -apple-system, 'Segoe UI', sans-serif", color=INK_2),
    legend=dict(orientation="h", x=0, y=-0.15, font=dict(size=12, color=INK_2)),
    margin=dict(t=100, l=50, r=130, b=70), height=420,
)
fig.update_yaxes(rangemode="tozero", ticksuffix=" min", showline=False, ticks="",
                 gridcolor=GRID, gridwidth=1, zeroline=False, tickfont=dict(size=12, color=MUTED))
fig.update_xaxes(showgrid=False, showline=True, linecolor=GRID, ticks="",
                 tickfont=dict(size=13, color=INK_2))

fig.show()
fig.write_image("../figs/07_median_length_by_weekday.svg", width=1200)

In [29]:
import numpy as np

SURFACE, INK, INK_2, MUTED = "#fcfcfb", "#0b0b0b", "#52514e", "#898781"
# Color gradient
RAMPS = {
    "member": ["#e3eefb", "#b6d3f4", "#86b4ec", "#2a78d6", "#1d5aa5", "#143f75", "#0b2748"],
    "casual": ["#fbe3e2", "#f5b5b4", "#ec8685", "#e34948", "#b52e2e", "#8a1f1f", "#5f1414"],
}

# 1 = Monday ... 7 = Sunday
DAYS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
GROUPS = [("member", "Members"), ("casual", "Casual riders")]

# % of each group's own trips in every day x hour cell
# (normalizing per group compares the shape of the pattern, not the volume)
share = (
    df.group_by("member_casual", "day_of_week", "start_hour")
    .agg(pl.len().alias("n"))
    .with_columns((pl.col("n") / pl.col("n").sum().over("member_casual") * 100).alias("pct"))
)

def to_matrix(group):
    m = np.zeros((7, 24))
    rows = share.filter(pl.col("member_casual") == group).select("day_of_week", "start_hour", "pct")
    for d, h, p in rows.iter_rows():
        m[d - 1, h] = p
    return m

mats = {g: to_matrix(g) for g, _ in GROUPS}
zmax = max(m.max() for m in mats.values())  # same scale on both panels

fig = make_subplots(
    rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.03,
    subplot_titles=[label for _, label in GROUPS],
)

for i, (g, label) in enumerate(GROUPS, start=1):
    ramp = RAMPS[g]
    fig.add_trace(
        go.Heatmap(
            z=mats[g], x=list(range(24)), y=DAYS,
            zmin=0, zmax=zmax,
            colorscale=[[k / (len(ramp) - 1), c] for k, c in enumerate(ramp)],
            xgap=2, ygap=2,                      # 2px surface gap between cells
            colorbar=dict(
                orientation="h", thickness=8, len=0.4, outlinewidth=0, ticksuffix="%",
                x=0.235 if i == 1 else 0.765, xanchor="center", y=-0.2, yanchor="top",
                title=dict(text="% of group's trips", side="top", font=dict(size=11, color=INK_2)),
                tickfont=dict(size=11, color=MUTED),
            ),
            hovertemplate=f"<b>{label}</b><br>%{{y}}, %{{x}}h<br>%{{z:.2f}}% of the group's trips<extra></extra>",
        ),
        row=1, col=i,
    )

fig.update_layout(
    title=dict(
        text=("<b>Members ride at weekday rush hours; casual riders on Saturday afternoons</b>"
              f"<br><span style='font-size:13px;color:{INK_2}'>"
              "Trips by day of week and start hour, as % of each group's total</span>"),
        x=0, xanchor="left", font=dict(size=18, color=INK),
    ),
    paper_bgcolor=SURFACE, plot_bgcolor=SURFACE,
    font=dict(family="system-ui, -apple-system, 'Segoe UI', sans-serif", color=INK_2),
    margin=dict(t=110, l=50, r=20, b=130), height=520,
)
fig.update_yaxes(autorange="reversed", showgrid=False, zeroline=False, showline=False,
                 ticks="", tickfont=dict(size=12, color=INK_2))
fig.update_xaxes(showgrid=False, zeroline=False, showline=False, ticks="",
                 tickmode="array", tickvals=list(range(0, 24, 3)),
                 ticktext=[f"{h}h" for h in range(0, 24, 3)],
                 tickfont=dict(size=12, color=INK_2),
                 title=dict(text="Start hour", font=dict(size=12, color=MUTED)))
fig.update_annotations(font=dict(size=14, color=INK), xanchor="left")
fig.show()
fig.write_image("../figs/08_heatmap_day_by_hour.svg", width=1200)